# NYC TLC Data Downloader
Parallel download helper for NYC TLC taxi trip and lookup data.

In [1]:
import os
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

In [2]:
# --- Configuration ---
YEAR = 2022
TAXI_TYPE = 'yellow'  # As per the project description
OUTPUT_DIR = '../raw'
# Hopefully static, would suck if it changed :/
BASE_URL = 'https://d37ci6vzurychx.cloudfront.net'

In [3]:
def download_file(url, local_path):
    """Downloads a file if it doesn't exist, with performance in mind."""
    if os.path.exists(local_path):
        print(f"Exists: {os.path.basename(local_path)}")
        return local_path

    try:
        print(f"Downloading: {os.path.basename(local_path)}")
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(local_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        return local_path
    except requests.exceptions.RequestException as e:
        print(f"Error downloading {url}: {e}")
        return None

In [4]:
def main():
    """Main execution function."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # --- Prepare URLs for download ---
    trip_urls = [f"{BASE_URL}/trip-data/{TAXI_TYPE}_tripdata_{YEAR}-{m:02d}.parquet" for m in range(1, 13)]
    all_urls = trip_urls + [f"{BASE_URL}/misc/taxi_zone_lookup.csv"]
    local_paths = [os.path.join(OUTPUT_DIR, os.path.basename(u)) for u in all_urls]

    # --- Execute Downloads in Parallel ---
    with ThreadPoolExecutor(max_workers=4) as executor:
        results = list(executor.map(download_file, all_urls, local_paths))

    if not all(results):
        print("\nSome files failed to download. Exiting.")
        return

    print("\n--- All downloads complete. ---")

In [5]:
if __name__ == '__main__':
    main()

Downloading: yellow_tripdata_2022-01.parquet
Downloading: yellow_tripdata_2022-02.parquet
Downloading: yellow_tripdata_2022-03.parquet
Downloading: yellow_tripdata_2022-04.parquet
Downloading: yellow_tripdata_2022-05.parquet
Downloading: yellow_tripdata_2022-06.parquet
Downloading: yellow_tripdata_2022-07.parquet
Downloading: yellow_tripdata_2022-08.parquet
Downloading: yellow_tripdata_2022-09.parquet
Downloading: yellow_tripdata_2022-10.parquet
Downloading: yellow_tripdata_2022-11.parquet
Downloading: yellow_tripdata_2022-12.parquet
Downloading: taxi_zone_lookup.csv

--- All downloads complete. ---
